# Retrain the MedGuard PHI model on MIMIC-III

Fine-tunes `obi/deid_roberta_i2b2` on MIMIC-III notes whose de-identification tags are replaced with fake values, once per seed. Then compares every retrained model with the original on held-out MIMIC patients and names and, optionally, on the n2c2 2014 test set, which neither model trained on.

**Set the runtime first:** Runtime > Change runtime type > GPU. An L4 or A100 is about 2-3x faster than a T4.

**MIMIC data rules**
- Check that this use fits your PhysioNet data use agreement. The default `bigquery` source reads PhysioNet's hosted copy under your own credentials. It needs a Google account linked to PhysioNet with BigQuery access enabled.
- Do not print or display note text, and do not save this notebook with note text in its output.
- The trained weights derive from MIMIC. Do not publish them or commit them. With `SAVE_TO_DRIVE` on, they are zipped to a private Drive folder so a disconnect does not lose finished seeds.

In [ ]:
# Settings
PROJECT_ID = ""  # Your Google Cloud project, used for BigQuery billing and quota
DATA_SOURCE = "bigquery"  # "bigquery" or "drive"
BQ_TABLE = "physionet-data.mimiciii_notes.noteevents"
DRIVE_FILE = "/content/drive/MyDrive/NOTEEVENTS.csv.gz"  # Only used when DATA_SOURCE is "drive"
CATEGORIES = ["Nursing/other", "Radiology", "Nursing", "Physician", "Discharge summary"]

SEEDS = [11, 12, 13]  # One training run per seed, about 45 minutes each on a T4
PER_CATEGORY = 1000  # Training notes per category
EPOCHS = 2
BATCH_SIZE = 16  # Use 8 on a T4 if you run out of memory
EVAL_PER_CATEGORY = 200
DETECTORS = "pipeline,known-patient-care"

# Optional external test: the n2c2 2014 de-identification test set with gold PHI tags,
# uploaded to this runtime through the Files panel. Leave empty to skip.
N2C2_ARCHIVE = ""  # For example "/content/testing-PHI-Gold-fixed.tar.gz" or a .zip
SAVE_TO_DRIVE = True  # Save each seed's model zip and results to Drive as it finishes, and resume from them
SAVE_MODELS_TO_DRIVE = True  # Set False to keep only the aggregate results in Drive
DRIVE_DIR = "/content/drive/MyDrive/medguard"  # Private Drive folder; check your data use agreement allows it

REPO_URL = "https://github.com/azaveri879/medguard-llm.git"
REPO_BRANCH = "add-gateway-and-eval"
NOTES_CSV = "/content/notes.csv"

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found. Set Runtime > Change runtime type > GPU."
print("GPU:", torch.cuda.get_device_name(0))

!pip install -q faker transformers==5.22.2
![ -d /content/medguard-llm ] || git clone -q --branch {REPO_BRANCH} {REPO_URL} /content/medguard-llm
%cd /content/medguard-llm/eval

In [ ]:
# Writes notes to this runtime's temporary disk. Prints a row count only.
import csv
import gzip
import shutil

csv.field_size_limit(10 ** 9)

if DATA_SOURCE == "bigquery":
    assert PROJECT_ID, "Set PROJECT_ID in the settings cell."
    from google.cloud import bigquery
    from google.colab import auth

    auth.authenticate_user()
    client = bigquery.Client(project=PROJECT_ID)
    query = f"""
        SELECT row_id, subject_id, TRIM(category) AS category, iserror, text
        FROM `{BQ_TABLE}`
        WHERE TRIM(category) IN UNNEST(@categories) AND text LIKE '%[**%'
    """
    config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("categories", "STRING", CATEGORIES)]
    )
    result = client.query(query, job_config=config).result()
    try:  # The Storage Read API streams Arrow batches in parallel, much faster than paging rows
        from google.cloud import bigquery_storage

        batches = result.to_arrow_iterable(bqstorage_client=bigquery_storage.BigQueryReadClient())
    except ImportError:
        batches = result.to_arrow_iterable()
    notes = []
    for batch in batches:
        columns = batch.to_pydict()
        notes.extend(zip(columns["row_id"], columns["subject_id"], columns["category"],
                         columns["iserror"], columns["text"]))
    # Batches arrive in no fixed order. Sorting by row_id makes the seeded samples the same on every download.
    notes.sort(key=lambda note: note[0])
    with open(NOTES_CSV, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["subject_id", "category", "iserror", "text"])
        for _, subject_id, category, iserror, text in notes:
            writer.writerow([subject_id, category, iserror or "", text])
    rows = len(notes)
    del notes
else:
    from google.colab import drive

    drive.mount("/content/drive")
    with gzip.open(DRIVE_FILE, "rb") as source, open(NOTES_CSV, "wb") as target:
        shutil.copyfileobj(source, target)
    with open(NOTES_CSV, newline="", encoding="utf-8", errors="replace") as handle:
        rows = sum(1 for _ in csv.reader(handle)) - 1

print(f"Notes available: {rows:,}")

In [ ]:
# Trains and scores one seed at a time, saving each to Drive as soon as it finishes.
# Training uses only patients in the train split and names from the train list.
# After a disconnect, re-run the cells above and this one: finished seeds are restored from Drive.
import shutil
from pathlib import Path

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    for sub in ("models", "results"):
        Path(DRIVE_DIR, sub).mkdir(parents=True, exist_ok=True)


def mimic_test(label, model):
    """Score one model on the held-out MIMIC patients and names. Output is aggregate only."""
    names = [f"mimic_{label}.md", f"mimic_{label}.json"]
    if SAVE_TO_DRIVE and all(Path(DRIVE_DIR, "results", n).exists() for n in names):
        for n in names:
            shutil.copy(Path(DRIVE_DIR, "results", n), Path("results", n))
        print(f"{label}: MIMIC results restored from Drive")
        return
    print(f"\n=== MIMIC test: {label}")
    !PHI_NER_MODEL={model} python deid_benchmark.py --mimic --split test --faker-names --per-category {EVAL_PER_CATEGORY} --detectors {DETECTORS} --output results/mimic_{label}.md --per-note-output results/mimic_{label}.json < {NOTES_CSV}
    if SAVE_TO_DRIVE:
        for n in names:
            shutil.copy(Path("results", n), Path(DRIVE_DIR, "results", n))
        print(f"{label}: results saved to Drive")


models = {"original": "obi/deid_roberta_i2b2"}
for seed in SEEDS:
    label, output = f"seed{seed}", f"models/seed{seed}"
    drive_zip = Path(DRIVE_DIR, "models", f"{label}.zip")
    if Path(output, "model.safetensors").exists():
        print(f"{label}: already trained")
    elif SAVE_TO_DRIVE and drive_zip.exists():
        !mkdir -p models && unzip -qo {drive_zip} -d models
        print(f"{label}: model restored from Drive")
    else:
        print(f"\n=== training {label}")
        !python train_deid.py --output-dir {output} --seed {seed} --per-category {PER_CATEGORY} --epochs {EPOCHS} --batch-size {BATCH_SIZE} < {NOTES_CSV}
        assert Path(output, "model.safetensors").exists(), f"{label}: training did not save a model"
        if SAVE_TO_DRIVE and SAVE_MODELS_TO_DRIVE:
            !cd models && zip -qr {drive_zip} {label}
            print(f"{label}: model saved to Drive")
    models[label] = str(Path(output).resolve())
    mimic_test(label, models[label])


In [ ]:
# Score the original model on the same held-out MIMIC patients and names.
mimic_test("original", models["original"])


In [ ]:
# Paired comparison with the original model on the same notes, plus the spread across seeds.
for detector in DETECTORS.split(","):
    runs = " ".join(f"results/mimic_seed{seed}.json:{detector}" for seed in SEEDS)
    print(f"\n##### MIMIC test, {detector}")
    !python deid_compare.py results/mimic_original.json:{detector} {runs} --labels ALL,NAME,AGE_OVER_89

In [ ]:
# External test on n2c2 2014 (different hospital system, human-made surrogate PHI).
# Use the test set only: the original model was trained on the n2c2 training set.
import shutil

if N2C2_ARCHIVE:
    shutil.rmtree("/content/n2c2", ignore_errors=True)
    shutil.unpack_archive(N2C2_ARCHIVE, "/content/n2c2")
    for label, model in models.items():
        print(f"\n=== n2c2 test: {label}")
        !PHI_NER_MODEL={model} python deid_benchmark.py --n2c2 /content/n2c2 --detectors {DETECTORS} --output results/n2c2_{label}.md --per-note-output results/n2c2_{label}.json
        if SAVE_TO_DRIVE:
            for n in (f"n2c2_{label}.md", f"n2c2_{label}.json"):
                shutil.copy(Path("results", n), Path(DRIVE_DIR, "results", n))
    for detector in DETECTORS.split(","):
        runs = " ".join(f"results/n2c2_seed{seed}.json:{detector}" for seed in SEEDS)
        print(f"\n##### n2c2 test, {detector}")
        !python deid_compare.py results/n2c2_original.json:{detector} {runs} --labels ALL,NAME,AGE_OVER_89
else:
    print("Skipped: set N2C2_ARCHIVE to run the external test.")

In [ ]:
# The runtime's disk is erased when the session ends, including the notes and local weights.
# The tables in eval/results are aggregate only (note hashes and counts) and safe to download.
from google.colab import files

!rm -f /content/results.zip && cd results && zip -q /content/results.zip mimic_* n2c2_* 2>/dev/null; true
if SAVE_TO_DRIVE:
    shutil.copy("/content/results.zip", Path(DRIVE_DIR, "results.zip"))
    print(f"Copied results.zip to {DRIVE_DIR}")
files.download("/content/results.zip")
